In [1]:
# ============================================================
# 04_transfer_finetune_B.ipynb
# Fine-tune source-trained model on target receiver domain B
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras

# -----------------------------
# Config
# -----------------------------
DATA_ROOT = Path("/home/tonyliao/Location")
BUILD_DIR = DATA_ROOT / "dataset_build_hybrid"
SOURCE_DIR = DATA_ROOT / "source_train_runs"
OUT_DIR = DATA_ROOT / "transfer_finetune_B_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 64
EPOCHS_STAGE1 = 40
EPOCHS_STAGE2 = 60
LR_STAGE1 = 5e-5
LR_STAGE2 = 1e-5

TARGET_T = 256
TARGET_S = None

SOURCE_MODEL_PATH = SOURCE_DIR / "hybrid_source_A_best.keras"

FREEZE_KEYWORDS_STAGE1 = [
    "amp_ssl_encoder",
    "amp_encoder_local",
    "phase_geometry_encoder",
]

UNFREEZE_EXCEPT_STAGE2 = [
    "coarse_xy_out",
]

# -----------------------------
# Load data
# -----------------------------
def load_npz(path: Path):
    obj = np.load(path, allow_pickle=True)
    return {k: obj[k] for k in obj.files}

B_finetune = load_npz(BUILD_DIR / "B_finetune.npz")
B_val      = load_npz(BUILD_DIR / "B_val.npz")

with open(BUILD_DIR / "label_map.json", "r", encoding="utf-8") as f:
    label_meta = json.load(f)

label_map = label_meta["label_map"]
inv_label_map = {int(k): v for k, v in label_meta["inv_label_map"].items()}
num_classes = len(label_map)

print("B_finetune:", len(B_finetune["amp_path"]))
print("B_val:", len(B_val["amp_path"]))
print("num_classes:", num_classes)

if len(B_finetune["amp_path"]) == 0:
    raise ValueError("B_finetune is empty.")
if not SOURCE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing source model: {SOURCE_MODEL_PATH}")

# -----------------------------
# Coordinates
# -----------------------------
CLASS_CENTER_MAP = {
    "Empty":       [0.0, 0.0],
    # "Center":      [3.0, 4.0],
    # "Corner":      [0.0, 8.0],
    "Corner_LeftDown":  [1.0, 7.0],
    "Corner_LeftUp":    [1.0, 1.0],
    "Corner_RightDown": [5.0, 7.0],
    "Corner_RightUp":   [5.0, 1.0],
    # "Empty_Pred":  [0.0, 0.0],
    "LeftDown":    [2.0, 6.0],
    # "LeftDown_Far":[1.5, 7.0],
    "LeftMid":     [2.0, 4.0],
    "LeftUp":      [2.0, 2.0],
    # "LeftUp_Near": [2.0, 2.5],
    # "LeftUp_Pred": [2.0, 2.0],
    "MiddleDown":  [3.0, 6.0],
    "MiddleUp":    [3.0, 2.0],
    "RightDown":   [4.0, 6.0],
    "RightMid":    [4.0, 4.0],
    "RightUp":     [4.0, 2.0],
    # "Wall":        [6.0, 5.0],
}

# -----------------------------
# Helpers
# -----------------------------
def ensure_3d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x[..., None]
    return x

def resize_to_target(x, target_t, target_s):
    x_tf = tf.convert_to_tensor(x, dtype=tf.float32)
    x_tf = tf.image.resize(x_tf, size=(target_t, target_s), method="bilinear")
    return x_tf.numpy().astype(np.float32)

def zscore_per_sample(x):
    mu = np.mean(x, axis=(0, 1), keepdims=True)
    sd = np.std(x, axis=(0, 1), keepdims=True) + 1e-6
    return ((x - mu) / sd).astype(np.float32)

def load_amp(path):
    x = np.load(str(path)).astype(np.float32)
    return ensure_3d(x)

def load_pha(path, ref_shape=None):
    if path is None or str(path) == "":
        if ref_shape is None:
            raise ValueError("pha_path missing and ref_shape is None")
        return np.zeros(ref_shape, dtype=np.float32)

    p = Path(str(path))
    if not p.exists():
        if ref_shape is None:
            raise ValueError(f"pha_path does not exist: {path}")
        return np.zeros(ref_shape, dtype=np.float32)

    x = np.load(str(p)).astype(np.float32)
    return ensure_3d(x)

sample_amp = load_amp(B_finetune["amp_path"][0])
if TARGET_S is None:
    TARGET_S = sample_amp.shape[1]

print("TARGET_T =", TARGET_T)
print("TARGET_S =", TARGET_S)

def preprocess_amp(path):
    x = load_amp(path)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def preprocess_pha(path, amp_shape=None):
    x = load_pha(path, ref_shape=amp_shape)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def onehot(num_classes, y):
    v = np.zeros((num_classes,), dtype=np.float32)
    v[int(y)] = 1.0
    return v

def xy_from_label(label_name):
    label_name = str(label_name)
    if label_name not in CLASS_CENTER_MAP:
        return np.zeros((2,), dtype=np.float32)
    return np.asarray(CLASS_CENTER_MAP[label_name], dtype=np.float32)

# -----------------------------
# Sequence
# -----------------------------
class HybridSequence(keras.utils.Sequence):
    def __init__(self, obj, batch_size=16, shuffle=True):
        self.obj = obj
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(obj["amp_path"]))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        inds = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]

        x_amp, x_pha = [], []
        y_presence, y_class = [], []
        y_coarse, y_delta, y_final = [], [], []
        sw_presence, sw_class = [], []
        sw_coarse, sw_delta, sw_final = [], [], []

        for i in inds:
            amp = preprocess_amp(self.obj["amp_path"][i])
            pha = preprocess_pha(self.obj["pha_path"][i], amp_shape=amp.shape)

            label_name = str(self.obj["anchor_label"][i])
            label_id = int(self.obj["label_id"][i])
            presence = int(self.obj["presence"][i])

            xy = xy_from_label(label_name) if presence == 1 else np.zeros((2,), dtype=np.float32)
            coarse_xy = xy.copy()
            delta_xy = np.zeros((2,), dtype=np.float32)

            x_amp.append(amp)
            x_pha.append(pha)

            y_presence.append([presence])
            y_class.append(onehot(num_classes, label_id))
            y_coarse.append(coarse_xy)
            y_delta.append(delta_xy)
            y_final.append(xy)

            sw_presence.append(1.0)
            sw_class.append(1.0)
            loc_w = 1.0 if presence == 1 else 0.0
            sw_coarse.append(loc_w)
            sw_delta.append(loc_w)
            sw_final.append(loc_w)

        x = {
            "amp_in": np.stack(x_amp).astype(np.float32),
            "pha_in": np.stack(x_pha).astype(np.float32),
        }
        y = {
            "presence_out": np.stack(y_presence).astype(np.float32),
            "class_out": np.stack(y_class).astype(np.float32),
            "coarse_xy_out": np.stack(y_coarse).astype(np.float32),
            "amp_delta_out": np.stack(y_delta).astype(np.float32),
            "final_xy_out": np.stack(y_final).astype(np.float32),
        }
        sw = {
            "presence_out": np.asarray(sw_presence, dtype=np.float32),
            "class_out": np.asarray(sw_class, dtype=np.float32),
            "coarse_xy_out": np.asarray(sw_coarse, dtype=np.float32),
            "amp_delta_out": np.asarray(sw_delta, dtype=np.float32),
            "final_xy_out": np.asarray(sw_final, dtype=np.float32),
        }
        return x, y, sw

train_seq = HybridSequence(B_finetune, batch_size=BATCH_SIZE, shuffle=True)
val_seq   = HybridSequence(B_val, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# Model load
# -----------------------------
model = keras.models.load_model(SOURCE_MODEL_PATH, compile=False)
model.summary()

# -----------------------------
# Loss / metrics
# -----------------------------
losses = {
    "presence_out": keras.losses.BinaryCrossentropy(),
    "class_out": keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
    "coarse_xy_out": keras.losses.Huber(delta=0.5),
    "amp_delta_out": keras.losses.Huber(delta=0.25),
    "final_xy_out": keras.losses.Huber(delta=0.5),
}

loss_weights = {
    "presence_out": 1.0,
    "class_out": 1.0,
    "coarse_xy_out": 1.0,
    "amp_delta_out": 0.25,
    "final_xy_out": 1.5,
}

metrics = {
    "presence_out": [
        keras.metrics.BinaryAccuracy(name="acc"),
        keras.metrics.Precision(name="prec"),
        keras.metrics.Recall(name="rec"),
    ],
    "class_out": [
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.TopKCategoricalAccuracy(k=2, name="top2"),
    ],
    "coarse_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "amp_delta_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "final_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
}

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=losses,
        loss_weights=loss_weights,
        metrics=metrics,
    )

# -----------------------------
# Freeze / unfreeze helpers
# -----------------------------
def freeze_by_keywords(model, freeze_keywords):
    for layer in model.layers:
        layer.trainable = True
        lname = layer.name.lower()
        for kw in freeze_keywords:
            if kw.lower() in lname:
                layer.trainable = False
                break

def unfreeze_all_except(model, exclude_names):
    exclude_names = set(exclude_names)
    for layer in model.layers:
        layer.trainable = False if layer.name in exclude_names else True

# -----------------------------
# Stage 1
# -----------------------------
freeze_by_keywords(model, FREEZE_KEYWORDS_STAGE1)
compile_model(model, LR_STAGE1)

callbacks_stage1 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_transfer_B_stage1_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "transfer_B_stage1_log.csv")),
]

history_stage1 = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks_stage1,
    verbose=1,
)

stage1_best_path = OUT_DIR / "hybrid_transfer_B_stage1_best.keras"
if stage1_best_path.exists():
    model = keras.models.load_model(stage1_best_path, compile=False)

# -----------------------------
# Stage 2
# -----------------------------
unfreeze_all_except(model, UNFREEZE_EXCEPT_STAGE2)
compile_model(model, LR_STAGE2)

callbacks_stage2 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_transfer_B_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "transfer_B_stage2_log.csv")),
]

history_stage2 = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks_stage2,
    verbose=1,
)

model.save(OUT_DIR / "hybrid_transfer_B_final.keras")

history_all = {
    "stage1": history_stage1.history,
    "stage2": history_stage2.history,
}

with open(OUT_DIR / "transfer_B_history.json", "w", encoding="utf-8") as f:
    json.dump(history_all, f, indent=2)

summary = {
    "protocol": "train_on_A__finetune_on_B__predict_unseen_locations",
    "source_model_path": str(SOURCE_MODEL_PATH),
    "single_receiver_input_only": True,
    "fused_AB_input": False,
    "train_split": "B_finetune",
    "val_split": "B_val",
    "excluded_from_training": "target_eval_unseen_locations",
    "B_finetune_windows": int(len(B_finetune["amp_path"])),
    "B_val_windows": int(len(B_val["amp_path"])),
    "batch_size": BATCH_SIZE,
    "epochs_stage1": EPOCHS_STAGE1,
    "epochs_stage2": EPOCHS_STAGE2,
    "lr_stage1": LR_STAGE1,
    "lr_stage2": LR_STAGE2,
    "target_t": TARGET_T,
    "target_s": TARGET_S,
    "num_classes": int(num_classes),
}

with open(OUT_DIR / "transfer_B_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved to:", OUT_DIR)
print(json.dumps(summary, indent=2))

2026-04-13 14:25:24.693626: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-13 14:25:24.699938: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776090324.707315 1568082 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776090324.709551 1568082 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776090324.715273 1568082 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

B_finetune: 3091
B_val: 771
num_classes: 13
TARGET_T = 256
TARGET_S = 41


I0000 00:00:1776090325.942903 1568082 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "hybrid_source_A_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ amp_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_ssl_encoder     │ (None, 256)       │  1,306,432 │ amp_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_feat (Dense)    │ (None, 128)       │     32,896 │ amp_ssl_encoder[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pha_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ phase_geometry_enc… │ (None, 128)       │  1,273,536 │ pha_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ amp_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ single_receiver_fe… │ (None, 256)       │          0 │ phase_geometry_e… │
│ (Concatenate)       │                   │            │ amp_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ presence_out        │ (None, 1)         │         65 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ amp_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ single_receiver_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delta_gate_fusion   │ (None, 257)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
│                     │                   │            │ presence_out[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 2)         │        130 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      8,256 │ delta_gate_fusio… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_delta_out       │ (None, 2)         │          0 │ dense_4[0][0]     │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_out (Dense)   │ (None, 13)        │      1,677 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delta_gate (Dense)  │ (None, 1)         │         33 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coarse_xy_out       │ (None, 2)         │         26 │ class_out[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scaled_delta        │ (None, 2)         │          0 │ amp_delta_out[0]… │
│ (Multiply)          │                   │            │ delta_gate[0][0]  │
├─────────────────────┼───────────────────┼────────────┼─────────────────

...


Epoch 1: val_loss improved from inf to 5.07721, saving model to /home/tonyliao/Location/transfer_finetune_B_runs/hybrid_transfer_B_stage1_best.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 24s 327ms/step - amp_delta_out_loss: 0.0097 - amp_delta_out_mae: 0.1165 - class_out_acc: 0.0714 - class_out_loss: 3.3454 - class_out_top2: 0.1269 - coarse_xy_out_loss: 0.6459 - coarse_xy_out_mae: 1.7855 - final_xy_out_loss: 0.6455 - final_xy_out_mae: 1.7846 - loss: 6.5474 - presence_out_acc: 0.9035 - presence_out_loss: 1.5849 - presence_out_prec: 0.9214 - presence_out_rec: 0.9789 - val_amp_delta_out_loss: 0.0069 - val_amp_delta_out_mae: 0.1011 - val_class_out_acc: 0.1323 - val_class_out_loss: 2.4767 - val_class_out_top2: 0.3048 - val_coarse_xy_out_loss: 0.5708 - val_coarse_xy_out_mae: 1.5423 - val_final_xy_out_loss: 0.5692 - val_final_xy_out_mae: 1.5387 - val_loss: 5.0772 - val_presence_out_acc: 0.9131 - val_presence_out_loss: 1.1188 - val_presence_out_prec: 0.9227 - val_presence_out_rec: 0.9888 - learning_rate:

2026-04-13 14:29:56.685205: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11191', 16 bytes spill stores, 16 bytes spill loads

2026-04-13 14:29:57.044649: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11191', 68 bytes spill stores, 68 bytes spill loads

2026-04-13 14:29:57.136555: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 408 bytes spill stores, 408 bytes spill loads

2026-04-13 14:29:57.192454: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 256 bytes spill stores, 256 bytes spill loads

2026-04-13 14:29:57.257806: I ex

32/49 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - amp_delta_out_loss: 0.0056 - amp_delta_out_mae: 0.0796 - class_out_acc: 0.8950 - class_out_loss: 0.6051 - class_out_top2: 0.9724 - coarse_xy_out_loss: 0.1752 - coarse_xy_out_mae: 0.3625 - final_xy_out_loss: 0.1737 - final_xy_out_mae: 0.3690 - loss: 1.0758 - presence_out_acc: 0.9932 - presence_out_loss: 0.0336 - presence_out_prec: 0.9927 - presence_out_rec: 1.0000

2026-04-13 14:30:07.766345: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 28 bytes spill stores, 32 bytes spill loads

2026-04-13 14:30:07.802708: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_10513', 8 bytes spill stores, 8 bytes spill loads

2026-04-13 14:30:07.872707: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11191', 28 bytes spill stores, 32 bytes spill loads

2026-04-13 14:30:07.876194: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_10513', 4 bytes spill stores, 4 bytes spill loads

2026-04-13 14:30:08.016564: I external/l

49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step - amp_delta_out_loss: 0.0054 - amp_delta_out_mae: 0.0796 - class_out_acc: 0.9016 - class_out_loss: 0.5507 - class_out_top2: 0.9752 - coarse_xy_out_loss: 0.1573 - coarse_xy_out_mae: 0.3583 - final_xy_out_loss: 0.1555 - final_xy_out_mae: 0.3649 - loss: 0.9694 - presence_out_acc: 0.9941 - presence_out_loss: 0.0279 - presence_out_prec: 0.9936 - presence_out_rec: 1.0000
Epoch 1: val_loss improved from inf to 0.39204, saving model to /home/tonyliao/Location/transfer_finetune_B_runs/hybrid_transfer_B_best.keras
49/49 ━━━━━━━━━━━━━━━━━━━━ 26s 323ms/step - amp_delta_out_loss: 0.0054 - amp_delta_out_mae: 0.0796 - class_out_acc: 0.9019 - class_out_loss: 0.5481 - class_out_top2: 0.9753 - coarse_xy_out_loss: 0.1564 - coarse_xy_out_mae: 0.3578 - final_xy_out_loss: 0.1546 - final_xy_out_mae: 0.3644 - loss: 0.9643 - presence_out_acc: 0.9941 - presence_out_loss: 0.0277 - presence_out_prec: 0.9937 - presence_out_rec: 1.0000 - val_amp_delta_out_loss: 0.0052 - val_